In [1]:
# Create a SparkSession.
# Read the full_names.csv file into a DataFrame.
# Define a Python function to get the initials from a full name.
# Register this function as a UDF.
# Apply the UDF to create a new initials column.
# Show the resulting DataFrame.

In [2]:
pip install 'pyarrow>=4.0.0'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 30.5 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, IntegerType
from pyspark.sql.functions import pandas_udf
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.pandas.functions import PandasUDFType
import pandas as pd

In [3]:
spark = SparkSession.builder.appName("Name").getOrCreate()

25/04/07 09:32:19 WARN Utils: Your hostname, Nikhils-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.0.90 instead (on interface en0)
25/04/07 09:32:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/07 09:32:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = spark.read.csv("full_name.csv", header = True)

In [5]:
df.head()

Row(full_name='Eleanor Davies')

In [6]:
df.printSchema()

root
 |-- full_name: string (nullable = true)



25/04/07 09:32:31 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [7]:
def initial_func(s):
    initials = s.strip().split(" ")
    res = ""
    for a in initials:
        res += a[0]
    return res

25/04/06 22:49:18 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [8]:
print(initial_func("Te Eleanor Davies Nik "))

TEDN


In [9]:
spark.udf.register("initial_udf", initial_func)

<function __main__.initial_func(s)>

In [10]:
initial_udf = udf(initial_func, StringType())

In [11]:
df_new = df.withColumn("Initials",initial_udf("full_name"))

In [12]:
df_new.show()

+---------------+--------+
|      full_name|Initials|
+---------------+--------+
| Eleanor Davies|      ED|
|Theodore Wilson|      TW|
| Hazel Martinez|      HM|
|  Owen Anderson|      OA|
| Clara Thompson|      CT|
|  Samuel Garcia|      SG|
|   Violet Perez|      VP|
| Josephine Hall|      JH|
|    Henry Young|      HY|
|   Stella Allen|      SA|
|  Gabriel Scott|      GS|
|    Naomi Green|      NG|
|    Isaac Baker|      IB|
|     Ruby Adams|      RA|
|     Caleb King|      CK|
|    Lily Wright|      LW|
|    Elijah Hill|      EH|
|Scarlett Nelson|      SN|
|  Daniel Carter|      DC|
| Grace Mitchell|      GM|
+---------------+--------+
only showing top 20 rows



In [13]:
def character_length(full_name):
    names = full_name.strip().split(" ")
    length = 0

    for name in names:
        length += len(name)
    return length

In [14]:
print(character_length(" Scarlett   Nelson  "))

14


In [15]:
spark.udf.register("length_udf",character_length)

<function __main__.character_length(full_name)>

In [16]:
length_udf = udf(character_length, IntegerType())

In [17]:
df_final = df_new.withColumn("Name_length",length_udf("full_name"))

In [18]:
df_final.show()

+---------------+--------+-----------+
|      full_name|Initials|Name_length|
+---------------+--------+-----------+
| Eleanor Davies|      ED|         13|
|Theodore Wilson|      TW|         14|
| Hazel Martinez|      HM|         13|
|  Owen Anderson|      OA|         12|
| Clara Thompson|      CT|         13|
|  Samuel Garcia|      SG|         12|
|   Violet Perez|      VP|         11|
| Josephine Hall|      JH|         13|
|    Henry Young|      HY|         10|
|   Stella Allen|      SA|         11|
|  Gabriel Scott|      GS|         12|
|    Naomi Green|      NG|         10|
|    Isaac Baker|      IB|         10|
|     Ruby Adams|      RA|          9|
|     Caleb King|      CK|          9|
|    Lily Wright|      LW|         10|
|    Elijah Hill|      EH|         10|
|Scarlett Nelson|      SN|         14|
|  Daniel Carter|      DC|         12|
| Grace Mitchell|      GM|         13|
+---------------+--------+-----------+
only showing top 20 rows



In [26]:
@pandas_udf(StringType(), functionType = PandasUDFType.SCALAR)
def upper_vectorized(full_name:pd.Series):
    return full_name.str.apply(lambda name: name.upper())

/Users/nikhilkumar/pyspark_env/lib/python3.13/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning: In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(


In [27]:
df_with_uppercase = df_final.withColumn("uppercase_full_name", upper_vectorized("full_name"))

In [31]:
#df_with_uppercase.head()

In [7]:
df.createOrReplaceTempView("Names")